# OWASP

> **Important:**
> This loads the data into OWASP Dependency Track, which is a tool to track vulnerabilities in software dependencies. You can install it locally using Docker, and then use the API to create projects and upload SBOMs. The notebook also retrieves the vulnerabilities and metrics for each project and saves them into CSV files.

# LOAD SBOMs into OWASP Dependency Track

In [39]:
sboms = sorted(Path("/Users/dmk6603/Documents/swdb_opensource/5-generate_sbom/sboms").glob("*.json"), 
               key=lambda f: f.stat().st_size, reverse=True)
print(f"SBOM più grande: {sboms[0].stem} ({sboms[0].stat().st_size / 1024:.0f} KB)")

SBOM più grande: FACEBOOK-React.cdx (21964 KB)


In [40]:
import requests
from pathlib import Path

BASE = "http://localhost:8081/api/v1"
H = {"X-Api-Key": "odt_ZT7N9T1C_7Z0bTOCNDQNrsJocC47gmSDhN5SuZ031"}

f = Path("/Users/dmk6603/Documents/swdb_opensource/5-generate_sbom/sboms/FACEBOOK-React.cdx.json")
proj = requests.put(f"{BASE}/project", headers={**H, "Content-Type": "application/json"},
    json={"name": f.stem, "version": "latest"}).json()
requests.post(f"{BASE}/bom", headers=H, files={"bom": f.open("rb")}, data={"project": proj["uuid"]})
print(f"✓ {f.stem}")

✓ FACEBOOK-React.cdx


In [41]:
import time
time.sleep(300)

vulns = requests.get(f"{BASE}/vulnerability/project/{proj['uuid']}", headers=H).json()
comps = requests.get(f"{BASE}/component/project/{proj['uuid']}?limit=1", headers=H)
total = comps.headers.get("X-Total-Count", "?")
print(f"Componenti: {total}, Vulnerabilità: {len(vulns)}")


Componenti: 7353, Vulnerabilità: 399


In [46]:
import requests, time
from pathlib import Path

BASE = "http://localhost:8081/api/v1"
H = {"X-Api-Key": "odt_C2ZnqluI_g5rf0y6v22wxn8FlU9CrIGuhZ2ywQZqZ"}
sbom_dir = Path("/Users/dmk6603/Documents/swdb_opensource/5-generate_sbom/sboms")

ok, fail = 0, 0
for f in sorted(sbom_dir.glob("*.json")):
    name = f.stem
    r = requests.get(f"{BASE}/project/lookup?name={name}&version=latest", headers=H)
    if r.status_code == 200:
        print(f"⏭ {name}")
        continue
    r = requests.put(f"{BASE}/project", headers={**H, "Content-Type": "application/json"},
        json={"name": name, "version": "latest"})
    if r.status_code not in (200, 201):
        print(f"✗ PROGETTO {name} → {r.status_code}")
        fail += 1
        continue
    proj = r.json()
    r2 = requests.post(f"{BASE}/bom", headers=H, files={"bom": f.open("rb")}, data={"project": proj["uuid"]})
    if r2.status_code == 200:
        print(f"✓ {name}")
        ok += 1
    else:
        print(f"✗ SBOM {name} → {r2.status_code}")
        fail += 1
    time.sleep(2)

print(f"\nDone: {ok} ok, {fail} falliti")

⏭ APACHE-Hive.cdx
⏭ AbanteCart-AbanteCart.cdx
⏭ Aerospike-Aerospike.cdx
⏭ Alkacon_Software_GmbH-OpenCms.cdx
⏭ Alluxio_Inc_-Alluxio.cdx
⏭ Ametys-Ametys.cdx
⏭ Angular-AngularJS.cdx
⏭ Ant_Design-Ant_Design.cdx
⏭ Apollo_GraphQL-Apollo_Client.cdx
⏭ Apple-CUPS.cdx
⏭ Apple-Swift.cdx
⏭ Apps-Tornado.cdx
⏭ Arrow_Digital-Hotcakes.cdx
⏭ BENDELL_HOLDINGS_LLC-govCMS.cdx
⏭ Backbonejs-Backbonejs.cdx
⏭ Basho-Basho_Riak.cdx
⏭ Bitcoin-Bitcoin.cdx
⏭ BloomReach-Hippo.cdx
⏭ Blue_River_Interactive_Group-Mura_CMS.cdx
⏭ Blue_Spire_Consulting_Inc_-Aurelia.cdx
⏭ Bolt-Bolt.cdx
⏭ BuddyPress-BuddyPress.cdx
⏭ CESANTA-Mongoose.cdx
⏭ CFEngine-CFEngine.cdx
⏭ CHEROKEE-Cherokee.cdx
⏭ CKSource_sp__z_o_o__sp_k_-CKEditor.cdx
⏭ COMPILER-PHP.cdx
⏭ Cake_Software_Foundation_Inc_-CakePHP.cdx
⏭ Calpont-InfiniDB.cdx
⏭ Canonical-Juju.cdx
⏭ Canonical-LXC.cdx
⏭ Cask_Data-Cask_Data_Application_Platform.cdx
⏭ Catalyst_Framework-Catalyst_Framework.cdx
⏭ Citrix_Systems_Inc-CloudStack.cdx
⏭ Civilized_Discourse_Construction_Kit_Inc_-Discou

In [47]:
r = requests.put(f"{BASE}/project", headers={**HEADERS, "Content-Type": "application/json"},
    json={"name": "test", "version": "latest"})
print(r.status_code, r.text)

401 


# DOWNLOAD all the vulnerabilities and metrics for each project and save into CSV files

In [48]:
import pandas as pd, requests

BASE = "http://localhost:8081/api/v1"
H = {"X-Api-Key": "odt_ZT7N9T1C_7Z0bTOCNDQNrsJocC47gmSDhN5SuZ031"}

# prendi tutti i progetti (paginazione)
projects = []
page = 1
while True:
    r = requests.get(f"{BASE}/project?limit=100&page={page}", headers=H).json()
    if not r: break
    projects += r
    page += 1

# estrai vulnerabilità per ogni progetto
rows = []
for p in projects:
    vulns = requests.get(f"{BASE}/vulnerability/project/{p['uuid']}", headers=H).json()
    for v in vulns:
        rows.append({
            "project": p["name"],
            "vuln_id": v.get("vulnId"),
            "source": v.get("source"),
            "severity": v.get("severity"),
            "cvssV3": v.get("cvssV3BaseScore"),
            "description": v.get("description", "")[:200]
        })

df = pd.DataFrame(rows)
df.to_csv("output/dtrack_vulns.csv", index=False)
print(f"{len(projects)} progetti, {len(rows)} vulnerabilità totali")

320 progetti, 2002 vulnerabilità totali


In [6]:
comp_rows = []
for p in projects:
    comps = requests.get(f"{BASE}/component/project/{p['uuid']}?limit=500", headers=H).json()
    for c in comps:
        comp_rows.append({
            "project": p["name"],
            "component": c.get("name"),
            "version": c.get("version"),
            "group": c.get("group"),
            "purl": c.get("purl")
        })

df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv("output/dtrack_components.csv", index=False)

In [ ]:
metrics_rows = []
for p in projects:
    m = requests.get(f"{BASE}/metrics/project/{p['uuid']}/current", headers=H).json()
    metrics_rows.append({
        "project": p["name"],
        "critical": m.get("critical", 0),
        "high": m.get("high", 0),
        "medium": m.get("medium", 0),
        "low": m.get("low", 0),
        "components": m.get("components", 0),
        "vulnerabilities": m.get("vulnerabilities", 0)
    })

df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv("output/dtrack_metrics.csv", index=False)

# CLEAN: Remove all the projects

In [33]:
import requests

BASE = "http://localhost:8081/api/v1"
H = {"X-Api-Key": "odt_n8xhjfgr_FfuF50feJvCEKm42OPsu8hTcz0lPsP0X"}

page = 1
while True:
    projects = requests.get(f"{BASE}/project?limit=100&page={page}", headers=H).json()
    if not projects: break
    for p in projects:
        r = requests.delete(f"{BASE}/project/{p['uuid']}", headers=H)
        print(r.status_code, p['name'])
    page = 1  # ricomincia da 1 perché la lista si accorcia

# Statistics

In [21]:
# /Users/dmk6603/Documents/swdb_opensource/6-dependency_track_owasp/output/dtrack_metrics.csv
# snippet
# project,critical,high,medium,low,components,vulnerabilities
# Alluxio_Inc_-Alluxio.cdx,2,10,2,0,498,14
# BuddyPress-BuddyPress.cdx,0,0,0,0,7,0

# /Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/MAIN.csv
# snippet
# repo_url,VendorName,Product,ProductCategory,Total Sites,US Sites,Total Enterprises,Enterprises in US,gh_stars,gh_forks,gh_open_issues,gh_watchers,gh_language,gh_created_at,gh_updated_at,gh_pushed_at,gh_default_branch,gh_license,gh_archived,gh_size_kb,gh_size_gb
# https://github.com/Alluxio/alluxio,Alluxio Inc.,Alluxio,Big Data Processing,9096,4153,2152,832,7173,2948,1039,433,Java,2012-12-21T17:43:46Z,2026-03-28T15:00:16Z,2025-04-29T16:46:58Z,main,Apache-2.0,False,205633,0.1961069107055664
# https://github.com/Ametys/runtime,Ametys,Ametys,Web Content Management Systems,535,0,311,0,1,0,0,2,CSS,2016-11-16T19:06:43Z,2016-11-17T09:26:18Z,2016-11-16T19:18:27Z,master,Apache-2.0,False,79746,0.07605171203613281


import pandas as pd
df_metrics = pd.read_csv("/Users/dmk6603/Documents/swdb_opensource/6-dependency_track_owasp/output/dtrack_metrics.csv")
df_main = pd.read_csv("/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/MAIN.csv")

# create new column in metrics file: product and vendor
df_metrics["product"] = df_metrics["project"].apply(lambda x: x.split(".cdx")[0].split("-")[-1])
df_metrics["vendor"] = df_metrics["project"].apply(lambda x: x.split(".cdx")[0].split("-")[0])
# replace _ with space in product and vendor
df_metrics["product"] = df_metrics["product"].str.replace("_", " ")
df_metrics["vendor"] = df_metrics["vendor"].str.replace("_", " ")

# replace space at the end of vendor with letter .
df_metrics["vendor"] = df_metrics["vendor"].apply(lambda x: x[:-1] + "." if x.endswith(" ") else x)

# merge df_metrics with df_main file where df_metrics.product == df_main.Product and df_metrics.vendor == df_main.VendorName
df_merged = pd.merge(df_metrics, df_main, left_on=["product", "vendor"], right_on=["Product", "VendorName"], how="left")

# save in MAIN file
df_merged.to_csv("/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/MAIN.csv", index=False)

# head
print(df_merged.head())


                                             project  critical  high  medium  low  components  vulnerabilities      product                                     vendor                                  repo_url                                 VendorName      Product              ProductCategory  Total Sites  US Sites  Total Enterprises  Enterprises in US  gh_stars  gh_forks  gh_open_issues  gh_watchers gh_language         gh_created_at         gh_updated_at          gh_pushed_at gh_default_branch  gh_license gh_archived  gh_size_kb  gh_size_gb
0                           Alluxio_Inc_-Alluxio.cdx         2    10       2    0         498               14      Alluxio                               Alluxio Inc.        https://github.com/Alluxio/alluxio                               Alluxio Inc.      Alluxio          Big Data Processing       9096.0    4153.0             2152.0              832.0    7173.0    2948.0          1039.0        433.0        Java  2012-12-21T17:43:46Z  2026-03-28T15

# Latex Overall Table

In [22]:
import pandas as pd

df = pd.read_csv("/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/MAIN.csv")

# Sort by Total Enterprises descending, take top 10
df = df.sort_values("Total Enterprises", ascending=False).head(10)

# Select and rename columns
df = df[["product", "vendor", "ProductCategory", "components",
         "vulnerabilities", "Total Enterprises", "gh_stars",
         "gh_forks", "gh_open_issues", "gh_watchers", "gh_license"]]
df = df.rename(columns={"components": "libraries"})

# Add overall row with mean of numerical columns
numeric_cols = df.select_dtypes(include="number").columns
mean_row = df[numeric_cols].mean()
overall = pd.DataFrame([["Overall", "", "", *[mean_row.get(c, "") for c in df.columns if c not in ["product", "vendor", "ProductCategory", "gh_license"]], ""]],
                        columns=df.columns)

# Cleaner approach
overall = {col: "" for col in df.columns}
overall["product"] = "Overall"
for col in numeric_cols:
    overall[col] = round(df[col].mean(), 2)
df = pd.concat([df, pd.DataFrame([overall])], ignore_index=True)

print(df.to_string(index=False))

      product                         vendor       ProductCategory  libraries  vulnerabilities  Total Enterprises  gh_stars  gh_forks  gh_open_issues  gh_watchers  gh_license
Apache Hadoop The Apache Software Foundation   Big Data Processing     1666.0            126.0           305735.0   15510.0    9206.0           124.0        965.0  Apache-2.0
 Apache Spark The Apache Software Foundation   Big Data Processing      483.0              1.0           228584.0   43056.0   29141.0           307.0       2001.0  Apache-2.0
 Apache Flink The Apache Software Foundation   Big Data Processing      666.0              5.0           159046.0   25906.0   13906.0           321.0        916.0  Apache-2.0
 Apache Kafka The Apache Software Foundation   Big Data Processing       63.0              0.0           130978.0   32226.0   15070.0           325.0       1047.0  Apache-2.0
      jPlayer                        jPlayer Software as a Service        0.0              0.0           127039.0    4607.0  

In [30]:
import time
from pathlib import Path

for f in sorted(Path("/Users/dmk6603/Documents/swdb_opensource/5-generate_sbom/sboms").glob("*.json")):
    name = f.stem
    # trova il progetto esistente
    r = requests.get(f"{BASE}/project/lookup?name={name}&version=latest", headers=H)
    if r.status_code == 200:
        uuid = r.json()["uuid"]
        requests.post(f"{BASE}/bom", headers=H,
            files={"bom": f.open("rb")}, data={"project": uuid})
        print(f"↻ {name}")
        time.sleep(2)

↻ APACHE-Hive.cdx
↻ AbanteCart-AbanteCart.cdx
↻ Aerospike-Aerospike.cdx
↻ Alkacon_Software_GmbH-OpenCms.cdx
↻ Alluxio_Inc_-Alluxio.cdx
↻ Ametys-Ametys.cdx
↻ Angular-AngularJS.cdx
↻ Ant_Design-Ant_Design.cdx
↻ Apollo_GraphQL-Apollo_Client.cdx
↻ Apple-CUPS.cdx
↻ Apple-Swift.cdx
↻ Apps-Tornado.cdx
↻ Arrow_Digital-Hotcakes.cdx
↻ BENDELL_HOLDINGS_LLC-govCMS.cdx
↻ Backbonejs-Backbonejs.cdx
↻ Basho-Basho_Riak.cdx
↻ Bitcoin-Bitcoin.cdx
↻ BloomReach-Hippo.cdx
↻ Blue_River_Interactive_Group-Mura_CMS.cdx
↻ Blue_Spire_Consulting_Inc_-Aurelia.cdx
↻ Bolt-Bolt.cdx
↻ BuddyPress-BuddyPress.cdx
↻ CESANTA-Mongoose.cdx
↻ CFEngine-CFEngine.cdx
↻ CHEROKEE-Cherokee.cdx
↻ CKSource_sp__z_o_o__sp_k_-CKEditor.cdx
↻ COMPILER-PHP.cdx
↻ Cake_Software_Foundation_Inc_-CakePHP.cdx
↻ Calpont-InfiniDB.cdx
↻ Canonical-Juju.cdx
↻ Canonical-LXC.cdx
↻ Cask_Data-Cask_Data_Application_Platform.cdx
↻ Catalyst_Framework-Catalyst_Framework.cdx
↻ Citrix_Systems_Inc-CloudStack.cdx
↻ Civilized_Discourse_Construction_Kit_Inc_-Discou

KeyboardInterrupt: 